In [1]:
!pip install numpy==1.26.4 roboticstoolbox-python spatialmath-python plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 MB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 kB 14.1 MB/s eta 0:00

In [3]:
import roboticstoolbox as rtb
import spatialmath as sm
import numpy as np
import plotly.graph_objects as go

# -----------------------------
# Helper: get joint positions
# -----------------------------
def get_joint_points(robot, q):
    pts = []

    # base
    pts.append(np.array(robot.base.t).astype(float).flatten())

    # link frames
    for i in range(robot.n):
        Ti = robot.fkine(q, end=robot.links[i])
        pts.append(np.array(Ti.t).astype(float).flatten())

    # true end-effector / tool frame
    Tee = robot.fkine(q)
    pts.append(np.array(Tee.t).astype(float).flatten())

    return np.array(pts)

# -----------------------------
# Load Panda robot
# -----------------------------
panda = rtb.models.Panda()
panda.q = panda.qr.copy()

# Time step
dt = 0.05

# -----------------------------
# Define square trajectory
# -----------------------------
Tstart = panda.fkine(panda.q)

waypoints = [
    Tstart * sm.SE3.Trans(-0.1, -0.1, 0.40),
    Tstart * sm.SE3.Trans(-0.1,  0.1, 0.40),
    Tstart * sm.SE3.Trans( 0.1,  0.1, 0.40),
    Tstart * sm.SE3.Trans( 0.1, -0.1, 0.40),
    Tstart * sm.SE3.Trans(-0.1, -0.1, 0.40)
]

# Store waypoint positions
wp_xyz = np.array([wp.t for wp in waypoints])

# -----------------------------
# Run control simulation
# -----------------------------
all_qs = [panda.q.copy()]
x_actual, y_actual, z_actual = [], [], []

for Tep in waypoints:
    arrived = False
    t_local = 0.0

    while (not arrived) and (t_local < 5.0):
        Tcur = panda.fkine(panda.q)

        v, arrived = rtb.p_servo(
            Tcur,
            Tep,
            gain=1.0,
            threshold=1e-3
        )

        qd = np.linalg.pinv(panda.jacobe(panda.q)) @ v
        panda.q = panda.q + qd * dt

        pos = panda.fkine(panda.q).t
        x_actual.append(pos[0])
        y_actual.append(pos[1])
        z_actual.append(pos[2])

        all_qs.append(panda.q.copy())

        t_local += dt

    # stop robot at waypoint
    panda.qd = np.zeros(panda.n)

# Convert to arrays
all_qs = np.array(all_qs)
traj = np.column_stack([x_actual, y_actual, z_actual])

# Precompute robot geometry for every frame
all_pts = [get_joint_points(panda, q) for q in all_qs]

# -----------------------------
# Build Plotly animation
# -----------------------------
frames = []

for k in range(len(all_pts)):
    pts = all_pts[k]

    frames.append(
        go.Frame(
            data=[
                # Robot
                go.Scatter3d(
                    x=pts[:, 0],
                    y=pts[:, 1],
                    z=pts[:, 2],
                    mode='lines+markers',
                    line=dict(width=8, color='red'),
                    marker=dict(size=4, color='red'),
                    name='Robot'
                ),

                # Actual trajectory up to current frame
                go.Scatter3d(
                    x=traj[:min(k, len(traj)), 0] if k > 0 else [],
                    y=traj[:min(k, len(traj)), 1] if k > 0 else [],
                    z=traj[:min(k, len(traj)), 2] if k > 0 else [],
                    mode='lines',
                    line=dict(width=5, color='blue'),
                    name='Actual trajectory'
                ),

                # Waypoints
                go.Scatter3d(
                    x=wp_xyz[:, 0],
                    y=wp_xyz[:, 1],
                    z=wp_xyz[:, 2],
                    mode='markers',
                    marker=dict(size=6, color='black'),
                    name='Waypoints'
                )
            ],
            name=str(k)
        )
    )

# Initial robot geometry
pts0 = all_pts[0]

# Axis limits
all_xyz = np.vstack(all_pts + [wp_xyz, traj])
xmin, ymin, zmin = all_xyz.min(axis=0) - 0.1
xmax, ymax, zmax = all_xyz.max(axis=0) + 0.1

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=pts0[:, 0],
            y=pts0[:, 1],
            z=pts0[:, 2],
            mode='lines+markers',
            line=dict(width=8, color='red'),
            marker=dict(size=4, color='red'),
            name='Robot'
        ),
        go.Scatter3d(
            x=[traj[0, 0]] if len(traj) > 0 else [],
            y=[traj[0, 1]] if len(traj) > 0 else [],
            z=[traj[0, 2]] if len(traj) > 0 else [],
            mode='lines',
            line=dict(width=5, color='blue'),
            name='Actual trajectory'
        ),
        go.Scatter3d(
            x=wp_xyz[:, 0],
            y=wp_xyz[:, 1],
            z=wp_xyz[:, 2],
            mode='markers',
            marker=dict(size=6, color='black'),
            name='Waypoints'
        )
    ],
    frames=frames
)

fig.update_layout(
    title='Panda square trajectory in Colab',
    scene=dict(
        xaxis_title='X [m]',
        yaxis_title='Y [m]',
        zaxis_title='Z [m]',
        xaxis=dict(range=[xmin, xmax]),
        yaxis=dict(range=[ymin, ymax]),
        zaxis=dict(range=[zmin, zmax]),
        aspectmode='data'
    ),
    updatemenus=[
        dict(
            type='buttons',
            showactive=True,
            buttons=[
                dict(
                    label='Play',
                    method='animate',
                    args=[None, {
                        'frame': {'duration': 50, 'redraw': True},
                        'fromcurrent': True
                    }]
                ),
                dict(
                    label='Pause',
                    method='animate',
                    args=[[None], {
                        'frame': {'duration': 0, 'redraw': False},
                        'mode': 'immediate',
                        'transition': {'duration': 0}
                    }]
                )
            ]
        )
    ]
)

print("Last drawn point:", all_pts[-1][-1])
print("True end-effector:", panda.fkine(panda.q).t)

fig.show()

Output hidden; open in https://colab.research.google.com to view.